# Predictive Maintenance — XGBoost with SMOTE & SHAP

This notebook performs **multi-class** predictive maintenance on the AI4I 2020 dataset.  
Key improvements over the original SVM baseline:

| Step | Change |
|------|--------|
| 1 | Updated imports — `xgboost`, `imblearn`, `shap`, extended `sklearn.metrics` |
| 2 | Swapped `MultiOutputClassifier(SVC(…))` → `XGBClassifier(objective='multi:softprob')` |
| 3 | Injected **SMOTE** on training data (with `sample_weight` fallback) |
| 4 | Added **Macro F1-Score** and **ROC-AUC (OVR)** alongside accuracy |
| 5 | Appended **SHAP TreeExplainer** summary plot for model explainability |

## Step 1 — Dependency & Import Updates

In [ ]:
# ── Core libraries ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import os
import re
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ── Scikit-learn ───────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight

# ── XGBoost (Step 1 — new import) ─────────────────────────────────
import xgboost as xgb
from xgboost import XGBClassifier

# ── SMOTE for class imbalance (Step 1 — new import) ───────────────
from imblearn.over_sampling import SMOTE

# ── SHAP for model explainability (Step 1 — new import) ──────────
import shap

# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports loaded successfully.")
print(f"  xgboost : {xgb.__version__}")
print(f"  shap    : {shap.__version__}")

## Data Loading & Feature Engineering

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────
LOCAL_PATH = 'ai4i2020.csv'
KAGGLE_PATH = '/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv'

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f"Loaded from local path: {LOCAL_PATH}")
elif os.path.exists(KAGGLE_PATH):
    df = pd.read_csv(KAGGLE_PATH)
    print(f"Loaded from Kaggle path: {KAGGLE_PATH}")
else:
    raise FileNotFoundError(
        f"Dataset not found at '{LOCAL_PATH}' or '{KAGGLE_PATH}'. "
        "Please place ai4i2020.csv in the notebook directory."
    )

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── Feature engineering (same as original) ────────────────────────
df['temperature_difference'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power'] = np.round(
    df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * np.pi / 60, 4
)

df.sample(5)

## Multi-Class Target Construction

The original notebook treated this as a **multi-label** problem (5 binary columns via `MultiOutputClassifier`).  
We now convert it into a single **multi-class** target so XGBoost can natively handle it with `objective='multi:softprob'`.

Mapping:  
- `0` → No Failure  
- `1` → TWF (Tool Wear Failure)  
- `2` → HDF (Heat Dissipation Failure)  
- `3` → PWF (Power Failure)  
- `4` → OSF (Overstrain Failure)  
- `5` → RNF (Random Failure)

In [ ]:
# ── Build multi-class target ──────────────────────────────────────
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_names = ['No Failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

def assign_failure_class(row):
    """Return the first active failure mode (1-5), or 0 for no failure."""
    for idx, col in enumerate(failure_cols, start=1):
        if row[col] == 1:
            return idx
    return 0

df['failure_class'] = df.apply(assign_failure_class, axis=1)

print("Class distribution:")
print(df['failure_class'].value_counts().sort_index())
print()
for i, name in enumerate(failure_names):
    count = (df['failure_class'] == i).sum()
    print(f"  {i} — {name}: {count} ({count / len(df) * 100:.1f}%)")

In [ ]:
# ── Prepare features (X) and target (y) ───────────────────────────
drop_cols = ['UDI', 'Product ID', 'Machine failure'] + failure_cols + ['failure_class']
X = df.drop(columns=drop_cols).copy()
y = df['failure_class'].copy()

# Encode 'Type' column: L=0, M=1, H=2
X['Type'] = X['Type'].map({'L': 0, 'M': 1, 'H': 2})

# ── Clean column names ────────────────────────────────────────────
# XGBoost rejects feature names containing [ ] or <
X.columns = [re.sub(r'[\[\]<>]', '', col).strip() for col in X.columns]

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Clean feature names: {X.columns.tolist()}")
X.head()

## Train-Test Split & Scaling

In [ ]:
# ── Split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2, stratify=y
)

print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
print(f"\nTrain class distribution:\n{y_train.value_counts().sort_index()}")
print(f"\nTest class distribution:\n{y_test.value_counts().sort_index()}")

In [ ]:
# ── Scale features ────────────────────────────────────────────────
feature_names = X.columns.tolist()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Keep as DataFrames for SHAP later
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_names, index=X_test.index)

print("Scaling complete.")
X_train_scaled.head()

## Step 3 — Fix Class Imbalance (SMOTE)

Apply **SMOTE** to the **training data only** — the test set stays untouched to prevent data leakage.  
If SMOTE fails (e.g., too few minority samples), we fall back to `compute_sample_weight('balanced', y_train)` and pass it to `fit()`.

In [ ]:
# ── Apply SMOTE (Step 3) ──────────────────────────────────────────
USE_SAMPLE_WEIGHTS = False  # Will flip to True if SMOTE fails
sample_weights = None

try:
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    print("✅ SMOTE applied successfully.")
    print(f"   Before SMOTE: {X_train_scaled.shape[0]} samples")
    print(f"   After  SMOTE: {X_train_resampled.shape[0]} samples")
    print(f"\n   Resampled class distribution:")
    print(f"   {pd.Series(y_train_resampled).value_counts().sort_index().to_dict()}")
except Exception as e:
    print(f"⚠️ SMOTE failed: {e}")
    print("   Falling back to sample_weight='balanced' for XGBoost.")
    USE_SAMPLE_WEIGHTS = True
    X_train_resampled = X_train_scaled
    y_train_resampled = y_train
    sample_weights = compute_sample_weight('balanced', y_train)

## Step 2 — Model Definition (XGBoost)

Replace `MultiOutputClassifier(SVC(…))` with a single **XGBClassifier** configured for multi-class classification.

In [ ]:
# ── Define XGBoost model (Step 2) ─────────────────────────────────
num_classes = len(y.unique())

xgb_model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    num_class=num_classes,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
)

print(f"XGBClassifier configured for {num_classes}-class classification.")
print(xgb_model)

In [ ]:
# ── Train the model ───────────────────────────────────────────────
fit_params = {}
if USE_SAMPLE_WEIGHTS:
    fit_params['sample_weight'] = sample_weights

xgb_model.fit(X_train_resampled, y_train_resampled, **fit_params)
print("✅ Model training complete.")

## Step 4 — Upgraded Evaluation Metrics

In [ ]:
# ── Predictions ───────────────────────────────────────────────────
y_pred = xgb_model.predict(X_test_scaled)
y_pred_proba = xgb_model.predict_proba(X_test_scaled)

# ── Classification Report ─────────────────────────────────────────
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(
    y_test, y_pred,
    target_names=failure_names,
    zero_division=0
))

# ── Confusion Matrix ──────────────────────────────────────────────
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)
cm = confusion_matrix(y_test, y_pred)
print(cm)

# ── Key Metrics (Step 4 — upgraded) ───────────────────────────────
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
except ValueError as e:
    # Can happen if a class has 0 test samples
    roc_auc = float('nan')
    print(f"\n⚠️ ROC-AUC could not be computed: {e}")

print("\n" + "=" * 70)
print("SUMMARY METRICS")
print("=" * 70)
print(f"  Accuracy        : {acc:.4f}")
print(f"  Macro F1-Score  : {macro_f1:.4f}")
print(f"  ROC-AUC (OVR)   : {roc_auc:.4f}")

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=failure_names,
    yticklabels=failure_names,
    ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — XGBoost Multi-Class')
plt.tight_layout()
plt.show()

## Step 5 — Model Explainability (SHAP)

Use `shap.TreeExplainer` (optimised for tree-based models) to compute SHAP values for the test set, then render a summary plot showing which sensor features drive predictions for each failure mode.

In [ ]:
# ── SHAP explainability (Step 5) ──────────────────────────────────
# NOTE: The installed SHAP _tree.py has been patched (lines 2104-2110)
# to handle XGBoost >=2.0 vector base_score format.
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_scaled)

print(f"✅ SHAP values computed for {X_test_scaled.shape[0]} test samples.")
if isinstance(shap_values, list):
    print(f"   Number of output classes: {len(shap_values)}")
    print(f"   Shape per class: {shap_values[0].shape}")
else:
    print(f"   SHAP values shape: {shap_values.shape}")

In [ ]:
# ── SHAP summary plot (all classes) ───────────────────────────────
print("SHAP Summary Plot — Feature Importance Across All Failure Modes")
print("=" * 70)

shap.summary_plot(
    shap_values,
    X_test_scaled,
    class_names=failure_names,
    plot_type='bar',
    show=True
)

In [ ]:
# ── SHAP detailed dot plot per class ──────────────────────────────
if isinstance(shap_values, list):
    for cls_idx, cls_name in enumerate(failure_names):
        if cls_idx < len(shap_values):
            print(f"\nSHAP values for class: {cls_name}")
            shap.summary_plot(
                shap_values[cls_idx],
                X_test_scaled,
                plot_type='dot',
                show=True
            )
else:
    # For binary or when shap returns a single array
    shap.summary_plot(
        shap_values,
        X_test_scaled,
        plot_type='dot',
        show=True
    )